#### Implement a simple version of a two-tower retrieval model in pseudocode. Include: (a) embedding lookups for user and item, (b) a similarity score (e.g., dot product), and (c) a training objective that could be used to train this model.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm  # for progress bars
import random

In [2]:
class TwoTowerModel(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64, hidden_dims=[128, 64], temperature=0.07, learnable_temperature=False):
        super().__init__()
        
        # Temperature scaling (for contrastive / softmax-based retrieval)
        # If learnable_temperature is True we parameterize log_temperature for stability
        if learnable_temperature:
            # store log_temperature as a parameter so temperature stays positive
            self.log_temperature = nn.Parameter(torch.log(torch.tensor(temperature, dtype=torch.float32)))
        else:
            self.register_buffer('fixed_temperature', torch.tensor(temperature, dtype=torch.float32))
            self.log_temperature = None
        
        # User tower
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.user_tower = self._create_tower(embedding_dim, hidden_dims)
        
        # Item tower
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        self.item_tower = self._create_tower(embedding_dim, hidden_dims)
        
    def _create_tower(self, input_dim, hidden_dims):
        layers = []
        prev_dim = input_dim
        
        for dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.ReLU(),
                nn.BatchNorm1d(dim),
                nn.Dropout(0.2)
            ])
            prev_dim = dim
            
        return nn.Sequential(*layers)
    
    def temperature(self):
        if self.log_temperature is not None:
            return torch.exp(self.log_temperature)
        else:
            return self.fixed_temperature
    
    def encode_user(self, user_id):
        # user_id: scalar tensor, or 1D tensor of shape (batch,)
        user_emb = self.user_embedding(user_id)
        # user_emb shape can be (embedding_dim,), (batch, embedding_dim)
        if user_emb.dim() == 1:
            # single user -> make batch dimension, run tower, then squeeze
            return self.user_tower(user_emb.unsqueeze(0)).squeeze(0)
        else:
            # batch of users -> (batch, embedding_dim) -> pass through tower
            return self.user_tower(user_emb)
    
    def encode_item(self, item_id):
        # item_id: can be (batch,), or (batch, n_negatives), or (n_items,)
        item_emb = self.item_embedding(item_id)
        # item_emb shapes:
        # - (embedding_dim,) for single item
        # - (batch, embedding_dim) for batch of items
        # - (batch, n_negatives, embedding_dim) for negative samples
        if item_emb.dim() == 1:
            # single item
            return self.item_tower(item_emb.unsqueeze(0)).squeeze(0)
        elif item_emb.dim() == 2:
            # (batch, embedding_dim)
            return self.item_tower(item_emb)
        elif item_emb.dim() == 3:
            # (batch, n_negatives, embedding_dim)
            b, nneg, emb = item_emb.shape
            item_flat = item_emb.view(b * nneg, emb)             # (b*nneg, emb)
            out_flat = self.item_tower(item_flat)                # (b*nneg, final_dim)
            out = out_flat.view(b, nneg, -1)                     # (b, nneg, final_dim)
            return out
        else:
            raise ValueError(f"Unexpected item_emb.dim()={item_emb.dim()}")
    
    def compute_similarity(self, user_vector, item_vector):
        # Compute dot product similarity over the final hidden dimension
        # Then scale by temperature: logits = (u · v) / temperature
        t = self.temperature()
        if item_vector.dim() == 2:
            # both are (batch, D) or (1, D) vs (n_items, D)
            logits = torch.sum(user_vector * item_vector, dim=-1) / t
            return logits
        elif item_vector.dim() == 3:
            # item_vector: (batch, n_negatives, D); user_vector: (batch, D) -> expand
            logits = torch.sum(user_vector.unsqueeze(1) * item_vector, dim=-1) / t
            return logits
        else:
            raise ValueError(f"Unexpected item_vector.dim()={item_vector.dim()}")
    
    def forward(self, user_ids, pos_item_ids, neg_item_ids):
        # Encode users (batch, D)
        user_vectors = self.encode_user(user_ids)
        
        # Encode positive and negative items
        pos_item_vectors = self.encode_item(pos_item_ids)   # (batch, D)
        neg_item_vectors = self.encode_item(neg_item_ids)   # (batch, n_neg, D)
        
        # Compute similarities (scaled by temperature)
        pos_similarities = self.compute_similarity(user_vectors, pos_item_vectors)
        neg_similarities = self.compute_similarity(user_vectors, neg_item_vectors)
        
        return pos_similarities, neg_similarities

In [3]:
# Training loop with BPR loss
def train_step(model, user_ids, pos_item_ids, neg_item_ids, optimizer):
    """
    Training step with BPR loss for implicit feedback.
    Args:
        user_ids: tensor of user IDs
        pos_item_ids: tensor of positive item IDs (items the user interacted with)
        neg_item_ids: tensor of negative item IDs (sampled items the user didn't interact with)
    """
    # Get similarities
    pos_sim, neg_sim = model(user_ids, pos_item_ids, neg_item_ids)
    
    # BPR loss: -log(sigmoid(pos_sim - neg_sim))
    loss = -torch.mean(torch.log(torch.sigmoid(pos_sim.unsqueeze(1) - neg_sim)))
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item()

In [4]:
def sample_training_batch(ratings_df, batch_id, batch_size, n_negatives, unique_items, probs=None):
    """
    Sample a batch of positive user-item interactions and negative samples
    """
    # Sample positive interactions
    
    pos_samples = ratings_df.iloc[batch_id*batch_size:(batch_id+1)*batch_size]
    user_ids = torch.tensor(pos_samples['user_id'].values - 1, dtype=torch.long)  # ensure long tensor
    pos_item_ids = torch.tensor(pos_samples['item_id'].values - 1, dtype=torch.long)  # ensure long tensor
    n_items = unique_items.shape[0]
    # Generate negative samples for each user
    neg_item_ids = []
    for user_id in pos_samples['user_id']:
        # Get items this user has interacted with
        user_items = set(ratings_df[ratings_df['user_id'] == user_id]['item_id'])
        # Sample from items the user hasn't interacted with
        neg_items = []
        while len(neg_items) < n_negatives:
            if probs is None:
                neg_item = unique_items[np.random.randint(0, n_items)]
            else:
                neg_item = np.random.choice(unique_items, size=1, replace=True, p=probs)[0]
            if neg_item not in user_items and neg_item - 1 not in neg_items:
                    neg_items.append(neg_item - 1)  # subtract 1 as IDs start from 1
        neg_item_ids.append(neg_items)
    
    neg_item_ids = torch.tensor(neg_item_ids, dtype=torch.long)  # ensure long tensor
    return user_ids, pos_item_ids, neg_item_ids

def train_model(model, optimizer, ratings_df, train_step_func, batch_size=64, n_negatives=5, n_epochs=10):
    """
    Train the two-tower model
    """
    model.train()
    n_batches = (len(ratings_df) + batch_size - 1) // batch_size
    unique_items = ratings_df['item_id'].unique()
    for epoch in range(n_epochs):
        total_loss = 0
        progress_bar = tqdm(range(n_batches), desc=f'Epoch {epoch+1}/{n_epochs}')
        
        for batch_id in progress_bar:
            # Sample batch
            user_ids, pos_item_ids, neg_item_ids = sample_training_batch(
                ratings_df, batch_id, batch_size, n_negatives, unique_items)
            
            # Training step
            loss = train_step_func(model, user_ids, pos_item_ids, neg_item_ids, optimizer)
            total_loss += loss
            
            # Update progress bar
            progress_bar.set_postfix({'loss': f'{loss:.4f}'})
        
        avg_loss = total_loss / n_batches
        print(f'Epoch {epoch+1}/{n_epochs}, Average Loss: {avg_loss:.4f}')

In [5]:
# Load the data
users = pd.read_csv('ml-100k/u.user', sep='|', header=None, 
                   names=['user_id', 'age', 'gender', 'occupation', 'zip_code'])
items = pd.read_csv('ml-100k/u.item', sep='|', header=None, encoding='latin-1',
                    names=['item_id', 'title', 'release_date', 'video_release_date', 
                          'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
ratings = pd.read_csv('ml-100k/u.data', sep='\t', header=None,
                     names=['user_id', 'item_id', 'rating', 'timestamp'])

In [7]:
random.seed(42)

# Training setup
batch_size = 64
n_negatives = 5  # number of negative samples per positive
n_epochs = 10
n_users = users['user_id'].nunique()
n_items = items['item_id'].nunique()
n_ratings = 10000

# Initialize model and optimizer
model = TwoTowerModel(n_users=n_users, n_items=n_items)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train the model
ratings_df = ratings.sample(n_ratings).reset_index(drop=True)
train_model(model, optimizer, ratings_df, train_step,
           batch_size=batch_size, 
           n_negatives=n_negatives, 
           n_epochs=n_epochs)

# Inference example: get embeddings for ANN search
user_id = ratings_df.iloc[0]['user_id']
print(f"\nTesting recommendations for user {user_id}:")
model.eval()  # Set model to evaluation mode
with torch.no_grad():
    # Get user embedding for search
    user_id = torch.tensor([user_id - 1], dtype=torch.long)  # ensure long tensor
    user_vector = model.encode_user(user_id)
    
    # Get all item embeddings (in practice, this would be pre-computed)
    # Process items in batches to avoid memory issues
    batch_size = 1000
    all_similarities = []
    
    for i in range(0, n_items, batch_size):
        batch_items = torch.arange(i, min(i + batch_size, n_items), dtype=torch.long)
        item_vectors = model.encode_item(batch_items)
        similarities = model.compute_similarity(user_vector, item_vectors)
        all_similarities.append(similarities)
    
    # Combine all similarities
    similarities = torch.cat(all_similarities)
    top_items = torch.argsort(similarities, descending=True)[:10]
    
    # Print recommendations
    print("\nTop 10 recommended items:")
    for i, item_idx in enumerate(top_items + 1, 1):  # add 1 to get original IDs
        title = items.loc[items['item_id'] == item_idx.item(), 'title'].iloc[0]
        print(f"{i}. {title}")

Epoch 1/10: 100%|██████████| 157/157 [00:08<00:00, 19.32it/s, loss=nan]


Epoch 1/10, Average Loss: nan


Epoch 2/10: 100%|██████████| 157/157 [00:08<00:00, 18.52it/s, loss=nan]


Epoch 2/10, Average Loss: nan


Epoch 3/10: 100%|██████████| 157/157 [00:08<00:00, 19.23it/s, loss=nan]


...

Epoch 8/10, Average Loss: nan


Epoch 9/10: 100%|██████████| 157/157 [00:07<00:00, 19.92it/s, loss=nan]


Epoch 9/10, Average Loss: nan


Epoch 10/10: 100%|██████████| 157/157 [00:07<00:00, 20.53it/s, loss=nan]

Epoch 10/10, Average Loss: nan

Testing recommendations for user 643:

Top 10 recommended items:
1. Toy Story (1995)
2. GoldenEye (1995)
3. Four Rooms (1995)
4. Get Shorty (1995)
5. Copycat (1995)
6. Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)
7. Twelve Monkeys (1995)
8. Babe (1995)
9. Dead Man Walking (1995)
10. Richard III (1995)


#### Given pre-trained user and item embedding matrices U and V, write a Python snippet using Faiss to build an index for the item vectors and then query the top-10 nearest neighbors for a given user’s embedding.

In [2]:
import faiss

In [3]:
# Example inputs (replace these with your real matrices)
# U: (n_users, d), V: (n_items, d)
# u: a single user embedding of shape (d,) or (1, d)
# item_ids: optional array mapping V rows -> original item ids (len == n_items)
U = np.random.randn(1000, 64).astype('float32')
V = np.random.randn(10000, 64).astype('float32')
item_ids = np.arange(V.shape[0])  # identity mapping

# Choose search type: 'cosine' or 'l2'
search_type = 'cosine'  
search_type = 'l2'
k = 10  # top-k results

if search_type == 'cosine':
    # Normalize vectors to unit length (cosine similarity via inner product)
    faiss.normalize_L2(V)
    faiss.normalize_L2(U)  # optional if querying many users; safe to normalize U too

    # Build an inner-product (IP) index
    index = faiss.IndexFlatIP(V.shape[1])      # exact search
    # For large catalogs use IndexHNSWFlat or IndexIVFFlat+PQ (see notes below)

elif search_type == 'l2':
    # Use Euclidean distance (L2)
    index = faiss.IndexFlatL2(V.shape[1])

else:
    raise ValueError("search_type must be 'cosine' or 'l2'")

# Add item vectors to index (ensure float32, C-contiguous)
V_add = np.ascontiguousarray(V.astype('float32'))
index.add(V_add)  # index.ntotal == n_items

# Query example: find top-k items for a given user u
user_idx = 0
u = U[user_idx:user_idx+1]  # shape (1, d), float32 and normalized if cosine

# Make sure query is float32 & contiguous
q = np.ascontiguousarray(u.astype('float32'))

# Run search
distances, indices = index.search(q, k)  # distances shape (1, k), indices shape (1, k)

# Map indices to item IDs (if you stored them)
top_item_indices = indices[0]            # row -> indices into V
top_scores = distances[0]

top_item_ids = item_ids[top_item_indices]

print("Top-k item indices:", top_item_indices)
print("Top-k item IDs:", top_item_ids)
print("Top-k scores:", top_scores)

Top-k item indices: [3817 3884 8635 4342 5519 4362 1255 2222 7764 5298]
Top-k item IDs: [3817 3884 8635 4342 5519 4362 1255 2222 7764 5298]
Top-k scores: [62.927265 63.395668 66.90537  67.35778  67.585144 68.87059  69.30339
 69.70263  69.77403  70.10449 ]


#### Prepare the training data for a deep retrieval model with a large implicit feedback dataset (user, item, timestamp of interaction). Generate user-positive pairs and negative samples, perhaps using recent interactions for the user as context.

When preparing implicit feedback data for deep retrieval models, we need to consider:

a. **Temporal Splitting**
   - Split by time to avoid future leakage
   - Use recent interactions as user context
   - Test on future interactions

b. **Positive Sampling**
   - For each user, use N most recent interactions as context
   - Use next interaction as positive example
   - Can also use sliding windows for more training pairs

c. **Negative Sampling**
   - Random sampling from uninteracted items
   - Popular items more likely to be true negatives
   - Can use time-based sampling (items from same time period)

d. **Additional Context**
   - Time of day, day of week
   - User's recent interaction sequence
   - Item co-occurrence patterns

In [32]:
def prepare_training_data(ratings_df, context_window=10, n_negatives=5, val_days=7):
    """
    Prepare training data for deep retrieval model with temporal splitting and contextual sampling.
    
    Args:
        ratings_df: DataFrame with columns [user_id, item_id, rating, timestamp]
        context_window: Number of recent interactions to use as user context
        n_negatives: Number of negative samples per positive interaction
        val_days: Number of days to use for validation
        
    Returns:
        train_data: List of (user_id, context_items, target_item, neg_items) tuples
        val_data: Similar structure but for validation
    """
    # Sort by user and timestamp
    ratings_df = ratings_df.sort_values(['user_id', 'timestamp'])
    
    # Convert timestamp to datetime if it's not already
    ratings_df['datetime'] = pd.to_datetime(ratings_df['timestamp'], unit='s')
    
    # Find split time (last date - val_days)
    max_date = ratings_df['datetime'].max()
    split_date = max_date - pd.Timedelta(days=val_days)
    
    # Split into train and validation
    train_df = ratings_df[ratings_df['datetime'] < split_date]
    val_df = ratings_df[ratings_df['datetime'] >= split_date]
    
    def prepare_user_sequences(user_df, all_items):
        """Helper to prepare sequences for one user"""
        sequences = []
        items = user_df['item_id'].values
        
        # For each position after context_window
        for i in range(context_window, len(items)):
            context = items[i-context_window:i]  # Last context_window items
            target = items[i]                    # Next item is target
            
            # Sample negative items (excluding items in context and target)
            seen_items = set(context) | {target}
            neg_items = []
            while len(neg_items) < n_negatives:
                neg = np.random.choice(all_items)
                if neg not in seen_items and neg not in neg_items:
                    neg_items.append(neg)
            
            sequences.append({
                'user_id': user_df['user_id'].iloc[0],
                'context': context.tolist(),
                'target': target,
                'negatives': neg_items,
                'timestamp': user_df['timestamp'].iloc[i]
            })
        return sequences
    
    # Get all possible items for negative sampling
    all_items = ratings_df['item_id'].unique()
    
    # Prepare sequences for each user
    train_sequences = []
    val_sequences = []
    
    for user_id, user_df in train_df.groupby('user_id'):
        if len(user_df) > context_window:
            train_sequences.extend(prepare_user_sequences(user_df, all_items))
    
    for user_id, user_df in val_df.groupby('user_id'):
        if len(user_df) > context_window:
            val_sequences.extend(prepare_user_sequences(user_df, all_items))
            
    return train_sequences, val_sequences

# Example usage
context_window = 5  # Use last 5 interactions as context
n_negatives = 10    # Sample 10 negative items per positive
val_days = 7        # Use last 7 days for validation

train_sequences, val_sequences = prepare_training_data(
    ratings, 
    context_window=context_window,
    n_negatives=n_negatives,
    val_days=val_days
)

# Print example sequence
example = train_sequences[0]
print("Example training sequence:")
print(f"User ID: {example['user_id']}")
print(f"Context (last {context_window} items): {example['context']}")
print(f"Target item: {example['target']}")
print(f"Negative items: {example['negatives']}")
print(f"Timestamp: {pd.to_datetime(example['timestamp'], unit='s')}")

print(f"\nTotal sequences - Train: {len(train_sequences)}, Val: {len(val_sequences)}")

Example training sequence:
User ID: 1
Context (last 5 items): [168, 172, 165, 156, 196]
Target item: 166
Negative items: [np.int64(1416), np.int64(1624), np.int64(1680), np.int64(1439), np.int64(1356), np.int64(527), np.int64(989), np.int64(1264), np.int64(1245), np.int64(695)]
Timestamp: 1997-09-22 22:01:17

Total sequences - Train: 93020, Val: 2122


In [48]:
# Create PyTorch Dataset for training
class ImplicitFeedbackDataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return {
            'user_id': seq['user_id'] - 1,  # Convert to 0-based indexing
            'context': torch.tensor([x - 1 for x in seq['context']], dtype=torch.long),
            'target': torch.tensor(seq['target'] - 1, dtype=torch.long),
            'negatives': torch.tensor([x - 1 for x in seq['negatives']], dtype=torch.long)
        }

# Create train and validation datasets
train_dataset = ImplicitFeedbackDataset(train_sequences)
val_dataset = ImplicitFeedbackDataset(val_sequences)

# Create data loaders with batching
batch_size = 32
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0  # Increase if using multiple CPU cores
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

# Print shapes from a batch
batch = next(iter(train_loader))
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"{k}: {v.shape}")


Batch shapes:
user_id: torch.Size([32])
context: torch.Size([32, 5])
target: torch.Size([32])
negatives: torch.Size([32, 10])


#### Aassume a deep retrieval model is deployed and running, update the system if new items are added daily. Include updating item embeddings, rebuilding or updating the ANN index, and ensuring minimal downtime for serving recommendations.

1) High-level plan (daily)

Offline: generate embeddings for newly added items (same model version & preprocessing used in production).

Validate new embeddings (shape, norms, NaNs).

Update item-id → index-id mapping and metadata store.

Update the ANN index:

Preferred: Incrementally add items if index supports it (HNSW, Flat, IndexIDMap wrappers).

If index requires retraining (IVF/PQ) or quality degrades, build a new index offline then atomically swap it in.

Warm-up and smoke-test the new index on a small traffic / canary.

Atomic swap to production index (pointer/file rename or API swap).

Monitor metrics, rollback if issues.

2) Important invariants to maintain

Keep a stable mapping from item_id (business id) → index_id (index row). Use IndexIDMap or maintain a mapping file.

Ensure embedding dimension and normalization method match production (e.g., L2-normalize for cosine/IP).

Keep new index validated (no NaNs, correct dtype float32).

Provide a fallback to the old index for fast rollback.

3) Short Faiss pseudocode examples

In [ ]:
# A) Incremental add (HNSW or Flat wrapped with ID map)
# Pros: very fast add; no full rebuild; minimal downtime.
# Cons: possible small quality drift; some indices don't support remove well.

# Suppose `V_new` shape (n_new, d) float32, and new_item_ids are their business IDs (int64)
# Existing index in memory: index (faiss index). It's wrapped in IndexIDMap2 or IndexIDMap.
# Example: index = faiss.IndexFlatIP(d); index = faiss.IndexHNSWFlat(d, M); index = faiss.IndexIDMap(index)
d = 64
V_new = np.random.randn(10, d).astype('float32')
new_item_ids = 10000 + np.arange(V_new.shape[0])  # identity mapping

# Normalize if using cosine (optional)
faiss.normalize_L2(V_new)

# Add with IDs (so we can map business IDs directly)
index.add_with_ids(np.ascontiguousarray(V_new), np.array(new_item_ids, dtype=np.int64))
# Persist index (optional quick save)
faiss.write_index(index, '/mnt/indexes/items_index_current.faiss')

In [ ]:
# B) Full offline rebuild + atomic swap (for IVF/PQ or periodic rebuild)
# Build new index in a worker, save to disk, then atomically swap file/pointer.

# Train + build new index
quantizer = faiss.IndexFlatL2(d)
nlist = 0
m = 64
nbits = 10
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
V_all = np.concatenate([V, V_new])
all_item_ids = np.concatenate([item_ids, new_item_ids]) 
index.train(np.ascontiguousarray(V_all))   # must call train for IVF/PQ
index.add_with_ids(np.ascontiguousarray(V_all), np.array(all_item_ids, dtype=np.int64))
faiss.write_index(index, '/mnt/indexes/items_index_new.faiss')

# Atomic swap on serving node (zero-downtime):

# Option A: service loads new index into memory and switches pointer
# in serving process
new_index = faiss.read_index('/mnt/indexes/items_index_new.faiss')
# warm-up queries if desired
current_index = 0
old_index = current_index
current_index = new_index  # atomic pointer swap in memory
# optionally release old_index after a grace period

# Option B: change symlink to point to new file and restart a pool of workers gracefully (or instruct processes to reload); use rolling restart with load-balancer draining.


4) Full end-to-end pseudocode (daily job)

Collect new items (item metadata, raw features) for Day D.

Preprocess features (tokenize/normalize, image features, etc.).

Run embedding model (batch or micro-batch) offline:

Use the same model checkpoint in production or a validated new checkpoint.

Validate embeddings (dtype float32, shape (n_new, d), no NaN/Inf).

Insert to item metadata store and append mapping file: item_id -> index_id (use business IDs if using IndexIDMap).
Decide update strategy:

a) If index type supports add:

Add embeddings to index (index.add_with_ids).

Persist index snapshot.

b) Else:

Build new full index offline (train if needed).

Save new index file.

Warm-up & smoke tests:

Issue read-only test queries (synthetic and a small sample of real queries).

Compare results (recall/score distribution) vs previous index for a sample of users.

Canary rollout:

Serve small fraction (e.g., 1-5%) of traffic from new index.

Monitor latency, recall, CTR (if live), error rates.

Promote to full production or rollback if anomalies.

Periodic full rebuild: schedule weekly/monthly rebuilds for IVF/PQ to avoid accumulation of many incremental updates degrading accuracy.

5) Deployment / availability patterns (minimize downtime)

In-memory swap: load new index into memory in parallel, then atomically replace the pointer used by query handler. This gives near-zero downtime.

Dual-index serving (blue/green):

Keep both old and new index in memory, route a fraction of traffic to the new one, then switch all traffic.

Rolling restart + symlink:

Write index to unique path, update symlink (atomic), and gracefully reload workers one-by-one.

Use an index service:

Run a separate microservice that holds the index; you can deploy a new instance with the new index behind the load-balancer and remove old instances.

6) Consistency: ID mapping & feature store

Use IndexIDMap2 so query return indices are the original business item IDs. That avoids re-mapping.

Ensure item metadata (title, url, features) is stored in a WAL’ed metadata store (e.g., key-value DB).

Update any precomputed cross-features used by the ranker for new items (or lazily compute on-demand).

7) Monitoring and validation

Structural checks: index.ntotal increased by expected count; no NaNs.

Functional tests: a) top-N recall on a validation set b) latency and QPS c) result sanity (no duplicates).

Business metrics: CTR/engagement vs control in canary.

Alerting: index load failures, memory spikes, increase in query latency, drop in recall.

8) Edge cases and extras

Deletes / retire items: maintain tombstone list; for IndexIDMap you can remove IDs in some indices or maintain a filter at query time.

ID reuse: never reuse item IDs immediately; keep a monotonic id or include versioning.

Model change: if you retrain the embedding model, you must re-embed all items and rebuild index (or accept serving heterogeneous embeddings temporarily—rare & dangerous). 

Prefer full re-embed + rebuild.

Large catalog updates: if new items are large proportion, schedule nightly full rebuilds rather than repeated incremental adds to maintain index quality.

GPU indexes: for faiss-gpu, create and transfer index accordingly; still do offline build and atomic load on GPU worker.

9) Example daily pipeline (compact)

Ingest new items into upstream DB.

Batch preprocess -> Embedding job (distributed).

Save embeddings to object storage with metadata (e.g., s3://bucket/items/day=YYYYMMDD/).

Run index_update_worker:

Download embeddings

Validate

If incrementalable: POST to index service /index/add (index service calls index.add_with_ids and persists)

Else: build full index offline, save index file

Smoke test new index

Promote via atomic swap or start new service instances

Monitor for 24–72 hours, then mark items as live

10) Quick checklist to implement now

Use IndexIDMap2 in Faiss to preserve business item IDs.

Always keep backups of previous index snapshot (fast rollback).

Validate embeddings before injecting.

Canary traffic small % before full rollout.

Schedule periodic full rebuilds for indices that degrade (IVF/PQ).

#### Suppose instead of the dot product at the end of the two-tower model, we want to introduce a temperature factor to control the diversity of the output. Search the literature and implement such a variant.

In [6]:
# InfoNCE loss (softmax over positive + negatives)
def info_nce_loss(pos_sim, neg_sim):
    """
    Compute InfoNCE / softmax cross-entropy loss given positive and negative logits.
    Assumes pos_sim and neg_sim are already scaled by temperature (i.e., logits).
    Args:
        pos_sim: Tensor of shape (batch,)
        neg_sim: Tensor of shape (batch, n_neg)
    Returns:
        scalar loss tensor
    """
    # Concatenate positive (as column 0) and negatives -> shape (batch, 1 + n_neg)
    logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
    # Labels: positive is at index 0
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    loss = F.cross_entropy(logits, labels)
    return loss


def train_step_info_nce(model, user_ids, pos_item_ids, neg_item_ids, optimizer):
    """Training step using InfoNCE (cross-entropy over pos+neg logits)."""
    pos_sim, neg_sim = model(user_ids, pos_item_ids, neg_item_ids)
    loss = info_nce_loss(pos_sim, neg_sim)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [13]:
# Quick example: run one InfoNCE step on the first batch (uses existing sample_training_batch)
unique_items = ratings_df['item_id'].unique()
user_ids_b, pos_item_ids_b, neg_item_ids_b = sample_training_batch(ratings_df, batch_id=0, batch_size=64, n_negatives=5, unique_items=unique_items)
loss_val = train_step_info_nce(model, user_ids_b, pos_item_ids_b, neg_item_ids_b, torch.optim.Adam(model.parameters(), lr=0.001))
print(f"InfoNCE training loss (one batch): {loss_val:.4f}")

InfoNCE training loss (one batch): 170.8421


In [19]:
import matplotlib.pyplot as plt

# --- Popularity-based negative sampling utils ---
def build_popularity_probs(ratings_df, unique_items, clip_min=1e-6):
    """Return a probability vector aligned with unique_items for sampling negatives by popularity.
    unique_items: numpy array of item IDs (original 1-based IDs)
    """
    counts = ratings_df['item_id'].value_counts().to_dict()
    probs = np.array([counts.get(int(it), 0) for it in unique_items], dtype=np.float64)
    # Smooth and normalize
    probs = probs + clip_min
    probs = probs / probs.sum()
    return probs

# --- Training with temperature logging & plotting ---
def train_model_with_logging(model, optimizer, ratings_df, train_step_func, batch_size=64, n_negatives=5, n_epochs=5, sampling='uniform'):
    """Train and record temperature per epoch. sampling: 'uniform'|'popularity'|'inbatch'
    If sampling == 'inbatch', train_step_func should be the in-batch training step and sample_training_batch is not used.
    """
    model.train()
    device = next(model.parameters()).device
    n_batches = (len(ratings_df) + batch_size - 1) // batch_size
    unique_items = ratings_df['item_id'].unique()

    # Precompute popularity probs if needed
    if sampling == 'popularity':
        probs = build_popularity_probs(ratings_df, unique_items)
    else:
        probs = None

    temps = []
    losses = []

    for epoch in range(n_epochs):
        total_loss = 0.0
        progress_bar = tqdm(range(n_batches), desc=f'Epoch {epoch+1}/{n_epochs}')
        for batch_id in progress_bar:
            if sampling == 'inbatch':
                # sample only positives for in-batch negatives
                pos_samples = ratings_df.iloc[batch_id*batch_size:(batch_id+1)*batch_size]
                user_ids = torch.tensor(pos_samples['user_id'].values - 1, dtype=torch.long)
                pos_item_ids = torch.tensor(pos_samples['item_id'].values - 1, dtype=torch.long)

                loss = train_step_func(model, user_ids, pos_item_ids, optimizer)
            else:
                user_ids, pos_item_ids, neg_item_ids = sample_training_batch(ratings_df, batch_id, batch_size, n_negatives, unique_items, probs)
                user_ids = user_ids.to(device)
                pos_item_ids = pos_item_ids.to(device)
                neg_item_ids = neg_item_ids.to(device)
                loss = train_step_func(model, user_ids, pos_item_ids, neg_item_ids, optimizer)

            total_loss += loss
            progress_bar.set_postfix({'loss': f'{loss:.4f}'})

        avg_loss = total_loss / max(1, n_batches)
        losses.append(avg_loss)
        current_temp = float(model.temperature().detach().cpu().numpy())
        temps.append(current_temp)
        print(f'Epoch {epoch+1}/{n_epochs}, Average Loss: {avg_loss:.4f}, Temperature: {current_temp:.4f}')

    # Plot temperature and loss
    fig, ax1 = plt.subplots(figsize=(6,3))
    ax1.plot(temps, label='temperature', color='C0')
    ax1.set_xlabel('epoch')
    ax1.set_ylabel('temperature', color='C0')
    ax1.tick_params(axis='y', labelcolor='C0')

    ax2 = ax1.twinx()
    ax2.plot(losses, label='loss', color='C1')
    ax2.set_ylabel('loss', color='C1')
    ax2.tick_params(axis='y', labelcolor='C1')

    ax1.set_title('Temperature and Loss over Epochs')
    fig.tight_layout()
    plt.show()

    return temps, losses

In [ ]:
# --- In-batch negatives InfoNCE ---
def train_step_info_nce_inbatch(model, user_ids, pos_item_ids, optimizer):
    """Compute InfoNCE using in-batch positives as negatives.
    user_ids: (batch,)
    pos_item_ids: (batch,)
    """
    device = next(model.parameters()).device
    user_ids = user_ids.to(device)
    pos_item_ids = pos_item_ids.to(device)

    # Get embeddings / vectors
    user_vectors = model.encode_user(user_ids)            # (B, D)
    pos_vectors = model.encode_item(pos_item_ids)         # (B, D)

    # Compute logits: (B, B) = user_vectors @ pos_vectors.T
    logits = torch.matmul(user_vectors, pos_vectors.t())
    logits = logits / model.temperature()

    # Cross-entropy with positive on the diagonal — much more efficient than sampling many negatives.
    labels = torch.arange(logits.size(0), dtype=torch.long, device=device)
    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

# --- Usage examples ---
# 1) Train with learnable temperature and log it using in-batch negatives (efficient)
model = TwoTowerModel(n_users=n_users, n_items=n_items, learnable_temperature=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# Move model to CPU/GPU as needed; here we keep on CPU

# Train for a few epochs using in-batch negatives
train_model_with_logging(model, optimizer, ratings_df, train_step_info_nce_inbatch, batch_size=64, n_negatives=0, n_epochs=3, sampling='inbatch')

In [20]:
# 2) Train with popularity-based negatives using InfoNCE (not in-batch)
model2 = TwoTowerModel(n_users=n_users, n_items=n_items, learnable_temperature=True)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
train_model_with_logging(model2, optimizer2, ratings_df, train_step_info_nce, batch_size=64, n_negatives=5, n_epochs=3, sampling='popularity')

Epoch 1/3:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 1/3: 100%|██████████| 157/157 [00:11<00:00, 13.13it/s, loss=126.3717]


Epoch 1/3, Average Loss: 149.8466, Temperature: 0.0809


Epoch 2/3: 100%|██████████| 157/157 [00:11<00:00, 13.43it/s, loss=49.9057] 


Epoch 2/3, Average Loss: 91.6392, Temperature: 0.0902


Epoch 3/3: 100%|██████████| 157/157 [00:11<00:00, 13.19it/s, loss=44.4922]

: 